# 03 · Which customers are risky?

**Goal:** find out which groups of customers fail to pay back most often.
**Main tool:** `groupby`, which splits the table into groups and calculates something for each group.

**Key idea:** `is_bad` is 1 for bad customers and 0 for good ones, so the **mean of `is_bad` = the share of bad customers** (the "bad rate").

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/german_credit_clean.csv")

# The overall bad rate, to compare every group against
overall = df["is_bad"].mean()
print(f"Overall bad rate: {overall:.0%}")

Overall bad rate: 30%


## A small helper function
We'll ask the same question for many columns ("what is the bad rate per group?"), so we write it **once** as a function instead of repeating the code.
- `groupby(column)`: split customers into groups
- `["is_bad"].agg(["mean", "count"])`: for each group, the bad rate and the number of customers
- `sort_values(..., ascending=False)`: riskiest group first

In [2]:
def bad_rate(column):
    """Bad rate and number of customers for every group in a column, riskiest first."""
    result = df.groupby(column, observed=True)["is_bad"].agg(["mean", "count"])
    result = result.rename(columns={"mean": "bad_rate", "count": "customers"})
    return result.sort_values("bad_rate", ascending=False).round(2)

## 1. Bank account (checking_status)

In [3]:
bad_rate("checking_status")

,bad_rate,customers
checking_status,,
< 0 DM,0.49,274
0-200 DM,0.39,269
>= 200 DM,0.22,63
no account,0.12,394


**What I see:** Customers with a **negative account (< 0 DM) are bad 49% of the time**, which is **4× more** than customers with no checking account (12%). The account balance is a strong warning sign.

## 2. Loan duration

In [4]:
bad_rate("long_loan")

,bad_rate,customers
long_loan,,
True,0.44,230
False,0.26,770


**What I see:** Loans **longer than 24 months are bad 44% of the time**, compared with 26% for shorter loans. Longer loans are riskier.

## 3. Age group

In [5]:
bad_rate("age_group")

,bad_rate,customers
age_group,,
19-25,0.42,190
26-35,0.30,398
51+,0.27,113
36-50,0.24,299


**What I see:** **Young customers (19–25) have the highest bad rate (42%)**, and 36–50-year-olds have the lowest (24%).

## 4. Savings

In [6]:
bad_rate("savings")

,bad_rate,customers
savings,,
< 100 DM,0.36,603
100-500 DM,0.33,103
unknown / none,0.17,183
500-1000 DM,0.17,63
>= 1000 DM,0.12,48


**What I see:** Customers with **less than 100 DM in savings are bad 36% of the time**, while those with 1,000 DM or more are bad only 13% of the time. More savings means lower risk.

## 5. Purpose of the loan

In [7]:
bad_rate("purpose")

,bad_rate,customers
purpose,,
education,0.44,50
other,0.42,12
new car,0.38,234
repairs,0.36,22
business,0.35,97
appliances,0.33,12
furniture/equipment,0.32,181
radio/TV,0.22,280
used car,0.17,103


**What I see:** Loans for **education (44%) and new cars (38%)** are the riskiest. **Used cars (17%) and radio/TV (22%)** are the safest. ⚠️ Some groups are very small (e.g. retraining: 9 customers), so their rates are not reliable. Always check the `customers` count.

## 6. Employment and housing

In [8]:
bad_rate("employment")

,bad_rate,customers
employment,,
< 1 year,0.41,172
unemployed,0.37,62
1-4 years,0.31,339
>= 7 years,0.25,253
4-7 years,0.22,174


In [9]:
bad_rate("housing")

,bad_rate,customers
housing,,
for free,0.41,108
rent,0.39,179
own,0.26,713


**What I see:** People employed **less than 1 year** are the riskiest (41%). **Homeowners (26%)** are much safer than people who rent (39%) or live for free (41%).

## 7. Credit history: a surprise

In [10]:
bad_rate("credit_history")

,bad_rate,customers
credit_history,,
no credits / all paid,0.62,40
all paid at this bank,0.57,49
paid on time so far,0.32,530
delays in the past,0.32,88
critical account,0.17,293


**What I see:** Surprisingly, customers with a **"critical account"** (loans at other banks) have the **lowest** bad rate (17%), while those who had **no credits or paid everything back** have the highest (62.5%, but only 40 customers).
A likely explanation: the bank was already careful with "critical" customers and only gave them loans when they were safe. **Data doesn't always match intuition, so check before you assume.**

## 8. Good vs bad customers: averages compared

In [11]:
df.groupby("is_bad")[["credit_amount", "duration", "monthly_payment", "age"]].mean().round(1)

,credit_amount,duration,monthly_payment,age
is_bad,,,,
0,2985.5,19.2,165.8,36.2
1,3938.1,24.9,172.0,34.0


**What I see:** Bad customers borrowed **more (3,938 vs 2,986 DM)**, for **longer (24.9 vs 19.2 months)**, and are slightly **younger** (34 vs 36). The monthly payment is almost the same, so the *size and length* of the loan matter more than the monthly burden.

## ⚖️ A note on fairness
The data also contains sex/marital status (`personal_status`) and `foreign_worker`. In the EU it is **not legal or fair** to decide on loans based on sex or nationality, so these columns should **not** be used as reasons to reject customers, even if they show differences in the data.

## Summary: the riskiest customer profile
1. **Negative bank account**: 49% bad (vs 30% overall). This is the strongest signal.
2. **Long loans (> 24 months)**: 44% bad.
3. **Young (19–25)**: 42% bad.
4. **Low savings (< 100 DM)**: 36% bad.
5. **Short employment (< 1 year)** and **renting**: around 40% bad.

These patterns tell us which features matter for the model in `04_model_from_scratch.ipynb`.